In [0]:
import requests, json
from datetime import datetime

def get_last_page(resource):
    if not spark.catalog.tableExists("fhir_catalog.bronze.page_tracker"):
        return 0
    
    df = spark.sql(f"""
        SELECT last_page FROM fhir_catalog.bronze.page_tracker
        WHERE resource = '{resource}'
    """)
    
    return df.collect()[0][0] if df.count() > 0 else 0


def update_last_page(resource, page):
    spark.sql(f"""
        MERGE INTO fhir_catalog.bronze.page_tracker t
        USING (SELECT '{resource}' AS resource, {page} AS last_page) s
        ON t.resource = s.resource
        WHEN MATCHED THEN UPDATE SET last_page = s.last_page
        WHEN NOT MATCHED THEN INSERT *
    """)


def ingest_fhir_incremental(resource, pages_to_load=60):

    base_url = f"https://hapi.fhir.org/baseR4/{resource}?_count=20"

    last_page = get_last_page(resource)
    start_page = last_page + 1

    next_url = base_url
    page = 1
    raw_data = []

    # skip loaded pages
    while page < start_page:
        response = requests.get(next_url)
        data = response.json()

        next_url = None
        for link in data.get("link", []):
            if link.get("relation") == "next":
                next_url = link.get("url")

        page += 1

    # now load new pages
    pages_loaded = 0

    while next_url and pages_loaded < pages_to_load:
        response = requests.get(next_url)
        data = response.json()

        raw_data.append({
            "api_url": next_url,
            "extraction_timestamp": datetime.now().isoformat(),
            "resource": resource,
            "data": json.dumps(data)
        })

        next_url = None
        for link in data.get("link", []):
            if link.get("relation") == "next":
                next_url = link.get("url")

        page += 1
        pages_loaded += 1

    print(f"{resource}: loaded {pages_loaded} pages starting from page {start_page}")

    if raw_data:
        df_raw = spark.createDataFrame(raw_data)

        load_date = datetime.now().strftime("%Y-%m-%d")

        path = f"/Volumes/fhir_catalog/raw/fhir_raw_vol/{resource.lower()}/json/load_date={load_date}/"

        df_raw.write.mode("append").json(path)

        # update tracker
        update_last_page(resource, last_page + pages_loaded)

In [0]:
ingest_fhir_incremental("Patient", pages_to_load=30)
ingest_fhir_incremental("Encounter", pages_to_load=30)
ingest_fhir_incremental("Observation", pages_to_load=30)
ingest_fhir_incremental("Condition", pages_to_load=30)

In [0]:
%sql
--  select * 
--  from   fhir_catalog.bronze.page_tracker

select * from fhir_catalog.audit.silver_log